# Predict Seagrass at different times
* Designed to work year by year
* Pick a location and run

In [1]:
import geopandas
import pathlib
import rioxarray
import numpy
import dask.distributed
import pandas
import rasterio

module_path = pathlib.Path.cwd().parent / 'scripts'
import sys
if str(module_path) not in sys.path:
    sys.path.append(str(module_path))
import utils
import sentinel2
import training
import sampling

%load_ext autoreload
%autoreload 2

In [2]:
cluster = dask.distributed.LocalCluster()
client = dask.distributed.Client(cluster)
display(client)

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:57220/status,
Dashboard: http://127.0.0.1:57220/status,Workers: 4
Total threads: 8,Total memory: 31.73 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:57222,Workers: 0
Dashboard: http://127.0.0.1:57220/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:57252,Total threads: 2
Dashboard: http://127.0.0.1:57253/status,Memory: 7.93 GiB
Nanny: tcp://127.0.0.1:57225,


# Value to edit

In [3]:
all_training_sites =  ["CatlinsLake", "CatlinsRiverMouth", "Childrens", "Duvauchelle", "Robinsons", "Takamatua", "Purau", "Ihutai",
              "IveyBay_Nov25", "IveyBay_Feb26", "LeftBank_Nov25", "LeftBank_Feb26", "Paremata_Nov25", "Paremata_Feb26",
              "Paremata_Feb25", "ThePoint_Nov25", "ThePoint_Feb26", "Takapuwahia_Nov25", "Takapuwahia_Feb26", "IveyBay_ThePoint_LeftBank_Oct24"]
large_seagrass_sites =  ["CatlinsRiverMouth", "Childrens", "Duvauchelle", "Robinsons", "Ihutai", "LeftBank_Nov25", "ThePoint_Nov25"]
large_sites =  ["CatlinsLake", "CatlinsRiverMouth", "Childrens", "Duvauchelle", "Robinsons", "Takamatua", "Ihutai", "LeftBank_Nov25", "LeftBank_Feb26", "ThePoint_Nov25", "ThePoint_Feb26"]


In [4]:
predict_max_cloud_covers = [1, 5, 0, 10] # percentage
predict_tide_deltas = {1: {"hrs": 0, "mins": 30}, 
                       2: {"hrs": 0, "mins": 45}, 
                       3: {"hrs": 1, "mins": 0}}

In [5]:
prediction_sites = all_training_sites
years = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]

In [6]:
data_path = utils.get_data_path()
utils.create_data_folders()


# Cells to run
* Get lowtide satellite images by year and site
* Predict Seagrass across satellite images

### Save satellite
The low-tide no cloud images across sites for a whole year

In [ ]:
for max_cloud_cover in predict_max_cloud_covers:
    for tide_delta in predict_tide_deltas.values():
        print(f"Max cloud percentage {max_cloud_cover}, Low tide delta {tide_delta['hrs']}hrs and {tide_delta['mins']}mins")
        predictions_path, satellite_path = utils.get_prediction_path(
            sample_method="sampling_2", method_2_threshold=0.5,
            model_max_cloud_cover=max_cloud_cover,
            predict_max_cloud_cover=max_cloud_cover,
            predict_low_tide_delta_hrs=tide_delta["hrs"],
            predict_low_tide_delta_mins=tide_delta["mins"],
            model_low_tide_delta_hrs=tide_delta["hrs"], # dummy value as models not considered this notebook
            model_low_tide_delta_mins=tide_delta["mins"]) # dummy value as models not considered this notebook
        
        for prediction_site in prediction_sites:
            for year in years:
                print(f"{prediction_site}: year {year}")
            
                satellite_file = satellite_path / f"{prediction_site}_{year}_sentinel-2.nc"
                site_polygon_file = utils.get_site_polygon_path(prediction_site)
            
                if not satellite_file.exists():
                    print(f"\tSave satellite image of the site {prediction_site} around lowtide without cloud in the year {year}")
                    site_polygon = geopandas.read_file(site_polygon_file)
                    satellite_data = sentinel2.get_low_tide_no_cloud_images_in_year(
                        geometry=site_polygon,
                        max_cloud_cover=max_cloud_cover,
                        year=year,
                        low_tide_delta_hrs=tide_delta["hrs"],
                        low_tide_delta_mins=tide_delta["mins"],
                    )
                    if len(satellite_data['time']) == 0:
                        print("Warning: No satellite data without cloud. Ignore")
                    else:
                        print("\t\tSave")
                        utils.write_netcdf_conventions_in_place(satellite_data)
                        satellite_data = satellite_data.rio.reproject(utils.CRS_NZTM)
                        satellite_data = satellite_data.rio.clip(site_polygon.geometry, drop=True, all_touched=True)
                        utils.write_netcdf_conventions_in_place(satellite_data)
                        utils.save_netcdf(satellite_data, satellite_file)
                    utils.rgb_from_satellite(satellite_data_path=satellite_file)

Max cloud percentage 1, Low tide delta 0hrs and 30mins
CatlinsLake: year 2016
	Save satellite image of the site CatlinsLake around lowtide without cloud in the year 2016
	Time from low tide: 0:24:27.970000 for S2A_MSIL2A_20160409T222532_R029_T59GLJ_20210211T024734
	Time from low tide: 0:05:02.026000 for S2A_MSIL2A_20160820T223902_R072_T59GLJ_20210212T162332
	Time from low tide: 0:04:42.026000 for S2A_MSIL2A_20161215T222542_R029_T59GLJ_20210213T212403
	Time from low tide: 0:07:32.030000 for S2A_MSIL2A_20160310T222532_R029_T59GLJ_20210528T132415
	Low tide tiles: 4 from a total lowish cloud cover tiles of 63
	No cloud tiles: 1 from the low tide tiles of 4
		Save
CatlinsLake: year 2017
	Save satellite image of the site CatlinsLake around lowtide without cloud in the year 2017
	Time from low tide: 0:29:58.974000 for S2A_MSIL2A_20170626T223701_R072_T59GLJ_20210210T015542
	Time from low tide: 0:07:28.974000 for S2A_MSIL2A_20171120T222531_R029_T59GLJ_20201014T210953
	Time from low tide: 0:20:0